%pip install wordninja

In [103]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from langdetect import detect
from deep_translator import GoogleTranslator
import wordninja

In [104]:
df = pd.read_csv('criminal_law_data .csv')
df.head()

,title,summary,content,links,url
0,Criminal law,Criminal lawis the body oflawthat relates tocr...,Actus reus\nMens rea\nCausation\nConcurrence\n...,"['/wiki/Wikipedia:Citing_sources', '/wiki/Insa...",https://en.wikipedia.org/wiki/Criminal_law
1,Criminal procedure,Criminal procedureis theadjudicationprocess of...,Fair trial\nPre-trial\nSpeedy trial\nJury tria...,"['/wiki/Right_to_development', '/wiki/Prosecut...",https://en.wikipedia.org/wiki/Criminal_procedure
2,Crime,NaN,NaN,['/wiki/Wikipedia:Protection_policy#semi'],https://en.wikipedia.org/wiki/Crime
3,International criminal law,International criminal law(ICL) is a body ofpu...,International criminal law(ICL) is a body ofpu...,"['/wiki/Category:Law', '/wiki/De_facto', '/wik...",https://en.wikipedia.org/wiki/International_cr...
4,Law & Order: Criminal Intent,\nLaw & Order: Criminal Intentis an Americanpo...,\nLaw & Order: CI\nCriminal Intent\nCI\nPolice...,"['/wiki/Wikipedia:Citing_sources', '/wiki/Alex...",https://en.wikipedia.org/wiki/Law_&_Order:_Cri...


In [105]:
df.shape

(10000, 5)

In [106]:
#check on missing values
missing_values = df.isnull()
missing_values.head()

,title,summary,content,links,url
0,False,False,False,False,False
1,False,False,False,False,False
2,False,True,True,False,False
3,False,False,False,False,False
4,False,False,False,False,False


In [107]:
#list of columns with missing values
for column in missing_values.columns.values.tolist():
    print(column)
    print (missing_values[column].value_counts())
    print("")

title
title
False    10000
Name: count, dtype: int64

summary
summary
False    9241
True      759
Name: count, dtype: int64

content
content
False    9246
True      754
Name: count, dtype: int64

links
links
False    10000
Name: count, dtype: int64

url
url
False    10000
Name: count, dtype: int64



In [108]:
# Drop rows where both columns 'content' and 'summary' are empty
df1=df
df_cleaned = df1.dropna(subset=['content', 'summary'], how='all')

In [109]:
#check on missing values
missing_values = df_cleaned.isnull()
missing_values.head()

,title,summary,content,links,url
0,False,False,False,False,False
1,False,False,False,False,False
3,False,False,False,False,False
4,False,False,False,False,False
5,False,False,False,False,False


In [110]:
#display the duplicate rows
duplicate_rows_df = df[df.duplicated()]
print("number of duplicate rows: ", duplicate_rows_df.shape)


number of duplicate rows:  (1408, 5)


In [111]:

df = df.drop_duplicates()

In [124]:
# as seen from the dataset above, we have to remove the \n from the summary and content column
# there are many \n in the content and the summary column
#this removes all the \n from the content table and summart
df_cleaned["content"] = df_cleaned["content"].str.replace("\n", " ", regex=True)
df_cleaned["summary"] = df_cleaned["summary"].str.replace("\n", " ", regex=True)

In [113]:
df.head()
df_cleaned.head()

,title,summary,content,links,url
0,Criminal law,Criminal lawis the body oflawthat relates tocr...,Actus reus Mens rea Causation Concurrence Acce...,"['/wiki/Wikipedia:Citing_sources', '/wiki/Insa...",https://en.wikipedia.org/wiki/Criminal_law
1,Criminal procedure,Criminal procedureis theadjudicationprocess of...,Fair trial Pre-trial Speedy trial Jury trial C...,"['/wiki/Right_to_development', '/wiki/Prosecut...",https://en.wikipedia.org/wiki/Criminal_procedure
3,International criminal law,International criminal law(ICL) is a body ofpu...,International criminal law(ICL) is a body ofpu...,"['/wiki/Category:Law', '/wiki/De_facto', '/wik...",https://en.wikipedia.org/wiki/International_cr...
4,Law & Order: Criminal Intent,Law & Order: Criminal Intentis an Americanpol...,Law & Order: CI Criminal Intent CI Police pro...,"['/wiki/Wikipedia:Citing_sources', '/wiki/Alex...",https://en.wikipedia.org/wiki/Law_&_Order:_Cri...
5,Criminal Law (film),Criminal Lawis a 1988 Americanlegalthrillerfi...,Gary Oldman Kevin Bacon Tess Harper Karen You...,"['/wiki/Mark_Kasdan', '/wiki/Thriller_film', '...",https://en.wikipedia.org/wiki/Criminal_Law_(film)


In [114]:
#remove columns links
# df_cleaned = df_cleaned.drop(columns=['links'])
df_cleaned.head()

,title,summary,content,links,url
0,Criminal law,Criminal lawis the body oflawthat relates tocr...,Actus reus Mens rea Causation Concurrence Acce...,"['/wiki/Wikipedia:Citing_sources', '/wiki/Insa...",https://en.wikipedia.org/wiki/Criminal_law
1,Criminal procedure,Criminal procedureis theadjudicationprocess of...,Fair trial Pre-trial Speedy trial Jury trial C...,"['/wiki/Right_to_development', '/wiki/Prosecut...",https://en.wikipedia.org/wiki/Criminal_procedure
3,International criminal law,International criminal law(ICL) is a body ofpu...,International criminal law(ICL) is a body ofpu...,"['/wiki/Category:Law', '/wiki/De_facto', '/wik...",https://en.wikipedia.org/wiki/International_cr...
4,Law & Order: Criminal Intent,Law & Order: Criminal Intentis an Americanpol...,Law & Order: CI Criminal Intent CI Police pro...,"['/wiki/Wikipedia:Citing_sources', '/wiki/Alex...",https://en.wikipedia.org/wiki/Law_&_Order:_Cri...
5,Criminal Law (film),Criminal Lawis a 1988 Americanlegalthrillerfi...,Gary Oldman Kevin Bacon Tess Harper Karen You...,"['/wiki/Mark_Kasdan', '/wiki/Thriller_film', '...",https://en.wikipedia.org/wiki/Criminal_Law_(film)


C:\Users\lish\AppData\Local\Temp\ipykernel_5872\2048498938.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['summary'] = df_cleaned['summary'].apply(lambda x: " ".join(wordninja.split(x)) if isinstance(x, str) else x)
C:\Users\lish\AppData\Local\Temp\ipykernel_5872\2048498938.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['content'] = df_cleaned['content'].apply(lambda x: " ".join(wordninja.split(x)) if isinstance(x, str) else x)
C:\Users\lish\AppData\Local\Temp\ipykernel_5872\

In [116]:
df_cleaned.head()

,title,summary,content,links,url
0,Criminal law,Criminal law is the body of law that relates t...,Act us reus Mens re a Causation Concurrence Ac...,"['/wiki/Wikipedia:Citing_sources', '/wiki/Insa...",https://en.wikipedia.org/wiki/Criminal_law
1,Criminal procedure,Criminal procedure is the adjudication process...,Fair trial Pre trial Speedy trial Jury trial C...,"['/wiki/Right_to_development', '/wiki/Prosecut...",https://en.wikipedia.org/wiki/Criminal_procedure
3,International criminal law,International criminal law ICL is a body of pu...,International criminal law ICL is a body of pu...,"['/wiki/Category:Law', '/wiki/De_facto', '/wik...",https://en.wikipedia.org/wiki/International_cr...
4,Law Order Criminal Intent,Law Order Criminal Intent is an American polic...,Law Order CI Criminal Intent CI Police procedu...,"['/wiki/Wikipedia:Citing_sources', '/wiki/Alex...",https://en.wikipedia.org/wiki/Law_&_Order:_Cri...
5,Criminal Law film,Criminal Law is a 1988 American legal thriller...,Gary Oldman Kevin Bacon Tess Harper Karen Youn...,"['/wiki/Mark_Kasdan', '/wiki/Thriller_film', '...",https://en.wikipedia.org/wiki/Criminal_Law_(film)


In [117]:


# Drop rows where 'Language' column is NaN
non_english_df = non_english_df.dropna(subset=['Language'])

# Define translation function
def translate_text(text):
    return GoogleTranslator(source='auto', target='en').translate(text)

# Apply translation
non_english_df['EnglishText'] = non_english_df['Language'].apply(translate_text)


In [118]:
#list of columns with missing values
for column in missing_values.columns.values.tolist():
    print(column)
    print (missing_values[column].value_counts())
    print("")

title
title
False    9246
Name: count, dtype: int64

summary
summary
False    9241
True        5
Name: count, dtype: int64

content
content
False    9246
Name: count, dtype: int64

links
links
False    9246
Name: count, dtype: int64

url
url
False    9246
Name: count, dtype: int64



In [119]:
#remove the missing value
df_cleaned = df_cleaned.dropna()

In [120]:
for column in missing_values.columns.values.tolist():
    print(column)
    print (missing_values[column].value_counts())
    print("")


title
title
False    9246
Name: count, dtype: int64

summary
summary
False    9241
True        5
Name: count, dtype: int64

content
content
False    9246
Name: count, dtype: int64

links
links
False    9246
Name: count, dtype: int64

url
url
False    9246
Name: count, dtype: int64



In [121]:
#remove urls
def remove_url(text):
    return re.sub(r'https?://\S+|www\.\S+', '', text)

In [122]:
#remove missing values at summary column
df_cleaned['summary'] = df_cleaned['summary'].apply(remove_url)

In [126]:
df_cleaned = df_cleaned.reset_index(drop=True)

In [129]:
df_cleaned.to_csv('crimninal_law_clean.csv')